# Đánh giá không gian Projected (t-SNE)
Notebook này dùng để trực quan hoá các vector nhúng (128-D) của không gian Projected.

In [8]:
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
from omegaconf import OmegaConf
from tqdm.notebook import tqdm

# Đảm bảo import được code trong thư mục src
sys.path.append(os.path.dirname(os.getcwd()))

from src.models.palm_model import ProbabilisticPalmModel
from src.models.unet_model import UNetPalmModel
from src.datasets.data_module import PalmDataModule
from src.engine.lightning_module import GenerativeLightningModule

from src.datasets import DatasetFactory
from src.engine.represent import optimize_r_in_projected_space

## 1. Tải Model & Cấu hình

In [11]:
# Cấu hình checkpoint & dataset
CHECKPOINT_PATH = r"E:\_drive\palm_logs\version_5\checkpoints\last.ckpt"
CONFIG_PATH = r"E:\_drive\palm_logs\version_5\config_backup.yaml"
DATASET_NAME = "own_original"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
config = OmegaConf.load(CONFIG_PATH)

# Khởi tạo DataLoader
datamodule = PalmDataModule(config)
datamodule.setup(stage="fit")
val_loader = datamodule.val_dataloader()

# Khởi tạo và nạp trọng số mô hình
model = GenerativeLightningModule(config)
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(checkpoint["state_dict"], strict=False)
model.to(device)
model.eval()
print("Model loaded successfully!")

--------------------------------------------------
[DATASET INFO] Tên Dataset: OwnOriginalDataset
[DATASET INFO] Thư mục: /content/data/own/collect
[DATASET INFO] Tổng số mẫu: 100
[DATASET INFO] Split Mode: Ratio
[DATASET INFO] Tập Train: 90 samples
[DATASET INFO] Tập Val: 10 samples
--------------------------------------------------
🚀 Sử dụng hàm mất mát đẩy/kéo (Metric Learning): ArcFaceLoss (dim=64, classes=100) 🚀


RuntimeError: Error(s) in loading state_dict for GenerativeLightningModule:
	size mismatch for contrastive_loss.loss_fn.W: copying a param with shape torch.Size([64, 231]) from checkpoint, the shape in current model is torch.Size([64, 100]).

## 2. Chuẩn bị Dataset

In [ ]:
# Khởi tạo dataset
_, gallery_loader, query_loader = DatasetFactory.create(DATASET_NAME, config, batch_size=32)

all_images = []
all_labels = []

# Lấy mẫu một lượng dữ liệu để visualize
for images, labels in gallery_loader:
    all_images.append(images)
    all_labels.append(labels)
    if len(torch.cat(all_images)) > 300: # Lấy khoảng 300 ảnh
        break

images = torch.cat(all_images).to(device)
labels = torch.cat(all_labels).cpu().numpy()

print(f"Extracted {len(images)} images for {len(set(labels))} identities.")

## 3. Trích xuất Vectors (Không tối ưu r vs Có tối ưu r)

In [ ]:
v_deterministic = []
v_optimized = []

with torch.no_grad():
    outputs = palm_model(images, decode=False)
    mu_batch = outputs['mu']
    logvar_batch = outputs['logvar']
    
    # Deterministic (projector trực tiếp từ mu)
    p_det = palm_model.projector(mu_batch)
    v_deterministic = torch.nn.functional.normalize(p_det, p=2, dim=1).cpu().numpy()

# Tối ưu R cho từng mẫu (tuỳ chọn, tốn thời gian)
print("Optimizing R for each image...")
r_optimized = []
for i in tqdm(range(len(mu_batch))):
    mu_c = mu_batch[i:i+1]
    logvar_c = logvar_batch[i:i+1]
    
    idx_others = [j for j in range(len(mu_batch)) if j != i]
    mu_others = mu_batch[idx_others]
    logvar_others = logvar_batch[idx_others]
    
    r_c = optimize_r_in_projected_space(
        mu_c, logvar_c, mu_others, logvar_others, 
        model=palm_model, device=device, config=OmegaConf.to_container(config, resolve=True), 
        num_samples=256, loss_type='bce', max_steps=50, lr=0.01, verbose=False
    )
    r_optimized.append(r_c.cpu().numpy())

v_optimized = np.array(r_optimized)
print("Extraction complete.")

## 4. Trực quan hoá bằng t-SNE

In [ ]:
# Hàm vẽ t-SNE
def plot_tsne(features, labels, title="t-SNE Visualization"):
    tsne = TSNE(n_components=2, perplexity=30, random_state=42)
    features_2d = tsne.fit_transform(features)
    
    plt.figure(figsize=(10, 8))
    sns.scatterplot(
        x=features_2d[:, 0], y=features_2d[:, 1],
        hue=labels,
        palette=sns.color_palette("hsv", len(set(labels))),
        legend='full',
        alpha=0.8
    )
    plt.title(title)
    plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)
    plt.tight_layout()
    plt.show()
    
    # Tính Silhouette Score
    score = silhouette_score(features, labels)
    print(f"Silhouette Score: {score:.4f} (Càng gần 1 càng tốt)")

# Vẽ Deterministic (Direct from MU)
plot_tsne(v_deterministic, labels, "t-SNE - Deterministic Projected (p = Projector(mu))")

# Vẽ Optimized (Tìm R)
plot_tsne(v_optimized, labels, "t-SNE - Optimized R trong Projected Space")